In [0]:
%sql
select * from gizmobox.bronze.v_orders

In [0]:
%sql
select value:order_id,value:items[0],* from gizmobox.bronze.v_orders

In [0]:
%sql
create or replace temporary view tv_orders_fixed as
select value,
regexp_replace(value,'"order_date": (\\d{4}-\\d{2}-\\d{2})','"order_date": "\$1"') as fixed_value
from gizmobox.bronze.v_orders

In [0]:
%sql
select schema_of_json(fixed_value) as schema,fixed_value from tv_orders_fixed limit 1

In [0]:
%sql
select from_json(fixed_value,'
STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as json_value,fixed_value from tv_orders_fixed

In [0]:
%sql
create or replace table gizmobox.silver.orders_json as 
select from_json(fixed_value,'
STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as json_value from tv_orders_fixed

In [0]:
%sql
select * from gizmobox.silver.orders_json

# Transform Orders data - Explode Arrays

In [0]:
%sql
select  
json_value.order_id,
json_value.order_status,
json_value.payment_method,
json_value.total_amount,
json_value.transaction_timestamp,
json_value.customer_id,
json_value.items
from gizmobox.silver.orders_json

In [0]:
%sql
create or replace temporary view tv_orders_exploded as
select  
json_value.order_id,
json_value.order_status,
json_value.payment_method,
json_value.total_amount,
json_value.transaction_timestamp,
json_value.customer_id,
explode(array_distinct(json_value.items)) as item
from gizmobox.silver.orders_json

In [0]:
%sql
select * from tv_orders_exploded

In [0]:
%sql
SELECT order_id,
order_status, payment_method, total_amount,
cast(transaction_timestamp as timestamp) as transaction_timestamp, customer_id, item.item_id, item.name, item.price, item.quantity, item.category, item.details.brand, item.details.color
FROM tv_orders_exploded;

In [0]:
%sql
create or replace table gizmobox.silver.orders as 
SELECT order_id,
order_status, payment_method, total_amount,
cast(transaction_timestamp as timestamp) as transaction_timestamp, customer_id, item.item_id, item.name, item.price, item.quantity, item.category, item.details.brand, item.details.color
FROM tv_orders_exploded;

In [0]:
%sql
select * from gizmobox.silver.orders